# Setup

In [1]:
# Load the API key from .env and set the feed URLs we will call
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from google.transit import gtfs_realtime_pb2

load_dotenv()
API_KEY = os.environ["TFNSW_API_KEY"]
HEADERS = {"Authorization": f"apikey {API_KEY}"}

TRIP_UPDATES_URL = "https://api.transport.nsw.gov.au/v2/gtfs/realtime/metro"
VEHICLE_POS_URL = "https://api.transport.nsw.gov.au/v2/gtfs/vehiclepos/metro"
ALERTS_URL = "https://api.transport.nsw.gov.au/v2/gtfs/alerts/metro"

# Trip Updates: fetch + decode

In [2]:
# Call the live trip updates feed and decode the protobuf bytes
resp = requests.get(TRIP_UPDATES_URL, headers=HEADERS)
print(resp.status_code, len(resp.content))

trip_feed = gtfs_realtime_pb2.FeedMessage()
trip_feed.ParseFromString(resp.content)

print(trip_feed.header)
print(len(trip_feed.entity))

200 49702
gtfs_realtime_version: "1.0"
incrementality: FULL_DATASET
timestamp: 1786780138

52


# Trip Updates: inspect a raw entity

In [3]:
# Look at one raw entity to see the real fields TfNSW sends
print(trip_feed.entity[0])

id: "0/2026-08-15T17:47:46+10:00/0241-001-104-014"
trip_update {
  trip {
    trip_id: "0241-001-104-014:1000"
    start_time: "16:57:00"
    start_date: "20260815"
    schedule_relationship: SCHEDULED
    route_id: "SMNW_M1"
    direction_id: 1
  }
  stop_time_update {
    stop_sequence: 1
    departure {
      delay: 0
      time: 1786777020
    }
    stop_id: "2155269"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 2
    arrival {
      delay: 0
      time: 1786777140
    }
    departure {
      delay: 25
      time: 1786777195
    }
    stop_id: "2155267"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 3
    arrival {
      delay: 0
      time: 1786777309
    }
    departure {
      delay: 28
      time: 1786777367
    }
    stop_id: "2155265"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 4
    arrival {
      delay: 4
      time: 1786777463
    }
    departure {
      delay: 27
  

# Trip Updates: flatten to DataFrame preview

In [4]:
# Flatten every stop time update into one row so we can read it as a table
rows = []
for entity in trip_feed.entity:
    tu = entity.trip_update
    for stu in tu.stop_time_update:
        rows.append({
            "entity_id": entity.id,
            "trip_id": tu.trip.trip_id,
            "route_id": tu.trip.route_id,
            "start_date": tu.trip.start_date,
            "stop_sequence": stu.stop_sequence,
            "stop_id": stu.stop_id,
            "arrival_time": stu.arrival.time,
            "arrival_delay": stu.arrival.delay,
            "departure_time": stu.departure.time,
            "schedule_relationship": gtfs_realtime_pb2.TripUpdate.StopTimeUpdate.ScheduleRelationship.Name(stu.schedule_relationship),
        })

trip_updates_df = pd.DataFrame(rows)
print(trip_updates_df.shape)
trip_updates_df.head(20)

(1092, 10)


,entity_id,trip_id,route_id,start_date,stop_sequence,stop_id,arrival_time,arrival_delay,departure_time,schedule_relationship
0,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,1,2155269,0,0,1786777020,SCHEDULED
1,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,2,2155267,1786777140,0,1786777195,SCHEDULED
2,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,3,2155265,1786777309,0,1786777367,SCHEDULED
3,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,4,2153402,1786777463,4,1786777516,SCHEDULED
4,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,5,2153404,1786777622,6,1786777676,SCHEDULED
5,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,6,2154264,1786777781,5,1786777832,SCHEDULED
6,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,7,2154262,1786777953,26,1786777983,SCHEDULED
7,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,8,2126159,1786778118,26,1786778148,SCHEDULED
8,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,9,2121225,1786778428,3,1786778495,SCHEDULED
9,0/2026-08-15T17:47:46+10:00/0241-001-104-014,0241-001-104-014:1000,SMNW_M1,20260815,10,2113351,1786778663,2,1786778693,SCHEDULED


In [5]:
# Check the time range to catch bad or missing arrival times
pd.to_datetime(trip_updates_df["arrival_time"], unit="s").describe()

count                   1092
mean     2023-12-04 13:20:15
min      1970-01-01 00:00:00
25%      2026-08-15 07:19:20
50%      2026-08-15 07:53:58
75%      2026-08-15 08:33:37
max      2026-08-15 09:47:14
Name: arrival_time, dtype: object

# Vehicle Positions: fetch + decode

In [6]:
# Call the live vehicle positions feed and decode the protobuf bytes
resp = requests.get(VEHICLE_POS_URL, headers=HEADERS)
print(resp.status_code, len(resp.content))

vehicle_feed = gtfs_realtime_pb2.FeedMessage()
vehicle_feed.ParseFromString(resp.content)

print(vehicle_feed.header)
print(len(vehicle_feed.entity))

200 8337
gtfs_realtime_version: "1.0"
incrementality: FULL_DATASET
timestamp: 1786780144

27


# Vehicle Positions: inspect a raw entity

In [7]:
# Look at one raw vehicle entity to see its real fields
print(vehicle_feed.entity[0])

id: "0/2026-08-15T07:48:54Z/RS019"
vehicle {
  trip {
    trip_id: "0241-001-151-015:1000"
    start_time: "17:15:00"
    start_date: "20260815"
    schedule_relationship: SCHEDULED
    route_id: "SMNW_M1"
    direction_id: 0
  }
  position {
    latitude: -33.779644
    longitude: 151.086746
    bearing: 298.66
    speed: 27
  }
  current_stop_sequence: 12
  current_status: IN_TRANSIT_TO
  timestamp: 1786780134
  congestion_level: CONGESTION
  stop_id: "2113352"
  vehicle {
    id: "RS019"
    label: "RS019"
    license_plate: "RS019"
  }
  occupancy_status: MANY_SEATS_AVAILABLE
}



# Vehicle Positions: flatten to DataFrame preview

In [8]:
# Flatten every vehicle into one row so we can read it as a table
rows = []
for entity in vehicle_feed.entity:
    v = entity.vehicle
    rows.append({
        "entity_id": entity.id,
        "trip_id": v.trip.trip_id,
        "route_id": v.trip.route_id,
        "vehicle_id": v.vehicle.id or entity.id,
        "vehicle_label": v.vehicle.label,
        "lat": v.position.latitude,
        "lon": v.position.longitude,
        "bearing": v.position.bearing,
        "speed": v.position.speed,
        "current_stop_sequence": v.current_stop_sequence,
        "current_status": gtfs_realtime_pb2.VehiclePosition.VehicleStopStatus.Name(v.current_status),
        "timestamp": v.timestamp,
        "occupancy_status": gtfs_realtime_pb2.VehiclePosition.OccupancyStatus.Name(v.occupancy_status) if v.HasField("occupancy_status") else None,
    })

vehicle_pos_df = pd.DataFrame(rows)
print(vehicle_pos_df.shape)
vehicle_pos_df.head(20)

(27, 13)


,entity_id,trip_id,route_id,vehicle_id,vehicle_label,lat,lon,bearing,speed,current_stop_sequence,current_status,timestamp,occupancy_status
0,0/2026-08-15T07:48:54Z/RS019,0241-001-151-015:1000,SMNW_M1,RS019,RS019,-33.779644,151.086746,298.660004,27.0,12,IN_TRANSIT_TO,1786780134,MANY_SEATS_AVAILABLE
1,1/2026-08-15T07:48:56Z/RS041,0241-001-102-015:1000,SMNW_M1,RS041,RS041,-33.885834,151.204987,32.990002,18.0,2,IN_TRANSIT_TO,1786780136,MANY_SEATS_AVAILABLE
2,2/2026-08-15T06:58:13Z/RS010,M1-O-SYD_UP-CUD_DN-1-20260815-060013:X,SMNW_M1,RS010,RS010,-33.691898,150.905334,249.869995,0.0,20,IN_TRANSIT_TO,1786777093,FEW_SEATS_AVAILABLE
3,3/2026-08-15T07:27:19Z/RS018,0241-001-115-008:1000,SMNW_M1,RS018,RS018,-33.914135,151.166656,231.910004,0.0,21,STOPPED_AT,1786778839,FEW_SEATS_AVAILABLE
4,4/2026-08-15T07:48:53Z/RS043,0241-001-101-015:1000,SMNW_M1,RS043,RS043,-33.853973,151.202728,14.260000,23.0,6,IN_TRANSIT_TO,1786780133,FEW_SEATS_AVAILABLE
5,5/2026-08-15T07:48:55Z/RS035,0241-001-106-014:1000,SMNW_M1,RS035,RS035,-33.798210,151.180969,179.080002,0.0,13,STOPPED_AT,1786780135,FEW_SEATS_AVAILABLE
6,6/2026-08-15T07:48:55Z/RS030,0241-001-109-014:1000,SMNW_M1,RS030,RS030,-33.692738,150.924973,145.130005,10.0,2,IN_TRANSIT_TO,1786780135,FEW_SEATS_AVAILABLE
7,7/2026-08-15T07:48:54Z/RS029,M1-O-SYD_UP-CUD_DN-1-20260815-070414:X,SMNW_M1,RS029,RS029,-33.728077,150.986694,249.710007,0.0,16,STOPPED_AT,1786780134,MANY_SEATS_AVAILABLE
8,8/2026-08-15T07:49:00Z/RS020,M1-O-SYD_UP-CUD_DN-1-20260815-072414:X,SMNW_M1,RS020,RS020,-33.795952,151.149338,257.730011,27.0,9,IN_TRANSIT_TO,1786780140,STANDING_ROOM_ONLY
9,9/2026-08-15T07:49:00Z/RS026,0241-001-110-013:1000,SMNW_M1,RS026,RS026,-33.690136,150.913589,258.480011,25.0,20,IN_TRANSIT_TO,1786780140,FEW_SEATS_AVAILABLE


In [9]:
# Check how fresh the vehicle timestamps are
pd.to_datetime(vehicle_pos_df["timestamp"], unit="s").describe()

count                     27
mean     2026-08-15 07:43:57
min      2026-08-15 06:58:13
25%      2026-08-15 07:48:51
50%      2026-08-15 07:48:54
75%      2026-08-15 07:48:57
max      2026-08-15 07:49:00
Name: timestamp, dtype: object

# Service Alerts: fetch + decode

In [10]:
# Call the live alerts feed and decode the protobuf bytes
resp = requests.get(ALERTS_URL, headers=HEADERS)
print(resp.status_code, len(resp.content))

alerts_feed = gtfs_realtime_pb2.FeedMessage()
alerts_feed.ParseFromString(resp.content)

print(alerts_feed.header)
print(len(alerts_feed.entity))

200 8718
gtfs_realtime_version: "2.0"
incrementality: FULL_DATASET
timestamp: 1786780072

2


# Service Alerts: inspect a raw entity

In [11]:
# Look at one raw alert to see its real fields
print(alerts_feed.entity[0] if alerts_feed.entity else "no active alerts")

id: "36b04344-8521-595d-be19-05bbf93c0a9e"
alert {
  active_period {
    start: 1786723200
    end: 1786896053
  }
  informed_entity {
    agency_id: "SMNW"
    route_id: "SMNW_M1"
    direction_id: 1
  }
  informed_entity {
    agency_id: "SMNW"
    route_id: "SMNW_M1"
    direction_id: 0
  }
  cause: MAINTENANCE
  effect: MODIFIED_SERVICE
  url {
    translation {
      text: "https://transportnsw.info/alerts/details#/ems-75874"
      language: "en"
    }
  }
  header_text {
    translation {
      text: "Train trackwork on the T4 Eastern Suburbs Line may affect how you travel"
      language: "en"
    }
  }
  description_text {
    translation {
      text: "Saturday 15 and Sunday 16 August \nTrain trackwork on the T4 Eastern Suburbs Line may affect how you travel.\nBuses replace trains between Bondi Junction and Central, but do not stop at Martin Place.\nIf travelling to T4 Eastern Suburbs & Illawarra Line stations change at:\n- Gadigal for express buses from Town Hall to Bondi Jun

# Service Alerts: flatten to DataFrame preview

In [12]:
# Flatten each alert into one row per affected route so we can read it as a table
rows = []
for entity in alerts_feed.entity:
    a = entity.alert
    header = a.header_text.translation[0].text if a.header_text.translation else None
    description = a.description_text.translation[0].text if a.description_text.translation else None
    route_ids = [ie.route_id for ie in a.informed_entity if ie.route_id] or [None]
    for route_id in route_ids:
        rows.append({
            "entity_id": entity.id,
            "cause": gtfs_realtime_pb2.Alert.Cause.Name(a.cause),
            "effect": gtfs_realtime_pb2.Alert.Effect.Name(a.effect),
            "header_text": header,
            "description_text": description,
            "route_id": route_id,
        })

alerts_df = pd.DataFrame(rows)
print(alerts_df.shape)
alerts_df.head(20)

(204, 6)


,entity_id,cause,effect,header_text,description_text,route_id
0,36b04344-8521-595d-be19-05bbf93c0a9e,MAINTENANCE,MODIFIED_SERVICE,Train trackwork on the T4 Eastern Suburbs Line...,Saturday 15 and Sunday 16 August \nTrain track...,SMNW_M1
1,36b04344-8521-595d-be19-05bbf93c0a9e,MAINTENANCE,MODIFIED_SERVICE,Train trackwork on the T4 Eastern Suburbs Line...,Saturday 15 and Sunday 16 August \nTrain track...,SMNW_M1
2,7e669567-64e5-5fec-83a2-7332090518c3,UNKNOWN_CAUSE,UNKNOWN_EFFECT,Station Update - Central,"From Sunday to Thursday, between 10pm and 5am,...",NSN_2a
3,7e669567-64e5-5fec-83a2-7332090518c3,UNKNOWN_CAUSE,UNKNOWN_EFFECT,Station Update - Central,"From Sunday to Thursday, between 10pm and 5am,...",NSN_2i
4,7e669567-64e5-5fec-83a2-7332090518c3,UNKNOWN_CAUSE,UNKNOWN_EFFECT,Station Update - Central,"From Sunday to Thursday, between 10pm and 5am,...",NSN_2k
5,7e669567-64e5-5fec-83a2-7332090518c3,UNKNOWN_CAUSE,UNKNOWN_EFFECT,Station Update - Central,"From Sunday to Thursday, between 10pm and 5am,...",WST_2c
6,7e669567-64e5-5fec-83a2-7332090518c3,UNKNOWN_CAUSE,UNKNOWN_EFFECT,Station Update - Central,"From Sunday to Thursday, between 10pm and 5am,...",WST_2d
7,7e669567-64e5-5fec-83a2-7332090518c3,UNKNOWN_CAUSE,UNKNOWN_EFFECT,Station Update - Central,"From Sunday to Thursday, between 10pm and 5am,...",IWL_2b
8,7e669567-64e5-5fec-83a2-7332090518c3,UNKNOWN_CAUSE,UNKNOWN_EFFECT,Station Update - Central,"From Sunday to Thursday, between 10pm and 5am,...",IWL_2c
9,7e669567-64e5-5fec-83a2-7332090518c3,UNKNOWN_CAUSE,UNKNOWN_EFFECT,Station Update - Central,"From Sunday to Thursday, between 10pm and 5am,...",IWL_2d


# Static GTFS: download combined NSW zip

In [13]:
# Download the full static schedule zip that covers all of NSW
import io
import zipfile

STATIC_GTFS_URL = "https://api.transport.nsw.gov.au/v1/publictransport/timetables/complete/gtfs"

resp = requests.get(STATIC_GTFS_URL, headers=HEADERS, timeout=180)
print(resp.status_code, len(resp.content))

static_zip = zipfile.ZipFile(io.BytesIO(resp.content))
print(static_zip.namelist())

200 297818554
['agency.txt', 'stops.txt', 'routes.txt', 'calendar.txt', 'calendar_dates.txt', 'shapes.txt', 'trips.txt', 'stop_times.txt', 'notes.txt', 'levels.txt', 'pathways.txt']


# Static GTFS: filter to Sydney Metro (agency_id SMNW)

In [14]:
# Keep only the routes, trips, stop times and stops that belong to Sydney Metro
with static_zip.open("routes.txt") as f:
    routes_df = pd.read_csv(f, dtype=str)
metro_routes_df = routes_df[routes_df.agency_id == "SMNW"]

with static_zip.open("trips.txt") as f:
    trips_df = pd.read_csv(f, dtype=str)
metro_trips_df = trips_df[trips_df.route_id.isin(metro_routes_df.route_id)]

metro_trip_ids = set(metro_trips_df.trip_id)
chunks = []
with static_zip.open("stop_times.txt") as f:
    for chunk in pd.read_csv(f, dtype=str, chunksize=200_000):
        matched = chunk[chunk.trip_id.isin(metro_trip_ids)]
        if not matched.empty:
            chunks.append(matched)
metro_stop_times_df = pd.concat(chunks, ignore_index=True)

with static_zip.open("stops.txt") as f:
    stops_df = pd.read_csv(f, dtype=str)
metro_stops_df = stops_df[stops_df.stop_id.isin(metro_stop_times_df.stop_id)]

print(metro_routes_df.shape, metro_trips_df.shape, metro_stop_times_df.shape, metro_stops_df.shape)

(1, 9) (1930, 11) (40442, 11) (42, 10)


# Static GTFS: preview routes/trips/stops

In [15]:
# Show a quick look at the filtered metro schedule data
display(metro_routes_df)
display(metro_trips_df.head())
display(metro_stops_df.head())

,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color,exact_times
3302,3-M1-sj2-1,SMNW,M1,M1 Metro North West and Bankstown Line,Sydney Metro Network,401,168388,FFFFFF,0


,route_id,service_id,trip_id,shape_id,trip_headsign,direction_id,block_id,wheelchair_accessible,route_direction,trip_note,bikes_allowed
105606,3-M1-sj2-1,TA+rs200+11,0241-001-101-002:1000,3-M1-sj2-1.4.R,Sydenham,1,NaN,1,Tallawong to Sydenham,NaN,NaN
105607,3-M1-sj2-1,TA+rs200+11,0241-001-101-003:1000,3-M1-sj2-1.8.H,Tallawong,0,NaN,1,Sydenham to Tallawong,NaN,NaN
105608,3-M1-sj2-1,TA+rs200+11,0241-001-101-004:1000,3-M1-sj2-1.4.R,Sydenham,1,NaN,1,Tallawong to Sydenham,NaN,NaN
105609,3-M1-sj2-1,TA+rs200+11,0241-001-101-005:1000,3-M1-sj2-1.9.H,Tallawong,0,NaN,1,Sydenham to Tallawong,NaN,NaN
105610,3-M1-sj2-1,TA+rs200+11,0241-001-101-006:1000,3-M1-sj2-1.5.R,Sydenham,1,NaN,1,Tallawong to Sydenham,NaN,NaN


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code
40,2000466,2000466,"Central Station, Platform 26",-33.88411728,151.20638050,NaN,200060,1,Level -3,NaN
41,2000467,2000467,"Central Station, Platform 27",-33.88414242,151.20643575,NaN,200060,1,Level -3,NaN
183,204471,204471,"Sydenham Station, Platform 1",-33.91370658,151.16719220,NaN,204420,1,NaN,1
184,204472,204472,"Sydenham Station, Platform 2",-33.91379507,151.16723974,NaN,204420,1,NaN,2
380,2121225,2121225,"Epping Station, Platform 5",-33.77287527,151.08227835,NaN,212110,1,Level -2,5
